Interactive 2D Spectra Viewer (G395H)\nThis notebook provides an interactive slider to scan through the 2D spectra images at **every wavelength point** across the entire IFU cube (3610 slices).

In [1]:
import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
import ipywidgets as widgets
from IPython.display import display

# Load background-subtracted IFU spectra for G395H
s3d_file = 'NIRSpec_IFU/COS-87259_BGSUB_g395h-f290lp_s3d.fits'
s3d = fits.open(s3d_file)
hdr = s3d['sci'].header
f3d = s3d['sci'].data

# Wavelength array (unit: micron)
wl0, dwl, wl_len = hdr['CRVAL3'], hdr['CDELT3'], hdr['NAXIS3']
wl = wl0 + np.arange(wl_len) * dwl

# We compute vmin and vmax for standard colormap clamping based on the entire cube
# Avoiding NaNs and extreme outliers
valid_data = f3d[~np.isnan(f3d)]
vmin = np.percentile(valid_data, 1)
vmax = np.percentile(valid_data, 99)

# Define clump parameters
clump_ids = ['Center', 'West', 'North', 'South', 'Southeast']
x = np.array([27.2994, 27.2994, 27.5317, 26.9051, 19.7599]) 
y = np.array([24.2436, 24.2436, 28.4938, 15.3950, 20.7517]) 
a = np.array([2.0362, 2.0362, 1.4309, 3.2745, 2.0370]) 
b = np.array([1.4896, 1.4896, 1.3812, 1.0209, 1.0339]) 
r = np.array([2.6471, 2.6471, 3.2883, 2.5689, 2.2962]) 
theta = np.array([-0.1644, -0.1644, -0.0619, -0.3366, 0.9059]) 


In [2]:
# Function to plot a single 2D slice
def plot_slice(index):
    plt.figure(figsize=(8, 6))
    
    # We plot the specific 2D image layer corresponding to the wavelength index
    img = plt.imshow(f3d[index], origin='lower', vmin=vmin, vmax=vmax, cmap='viridis')
    plt.colorbar(img, label='Flux')
    
    # Plot the elliptical aperture for each clump
    for n, clump_id in enumerate(clump_ids):
        e = Ellipse(xy=(x[n], y[n]), width=2.*a[n]*r[n], height=2.*b[n]*r[n], 
                    angle=theta[n]*180./np.pi, facecolor='none', edgecolor='red', lw=1.5)
        plt.gca().add_patch(e)
        plt.text(x[n] + 1.5, y[n] + 1.5, clump_id, color='red', fontsize=10)
    
    plt.title(f"G395H 2D Spectra at {wl[index]:.4f} $\mu$m (Slice {index}/{wl_len-1})")
    plt.xlabel('X (pixels)')
    plt.ylabel('Y (pixels)')
    plt.show()

# Create an interactive slider widget connecting to the plot function
slider = widgets.IntSlider(
    value=wl_len // 2,        # Start in the middle of the cube
    min=0,
    max=wl_len - 1,
    step=1,
    description='Slice:',
    continuous_update=False,  # Wait to update the plot until mouse release
    layout=widgets.Layout(width='80%')
)

display(widgets.interactive(plot_slice, index=slider))


interactive(children=(IntSlider(value=1805, continuous_update=False, description='Slice:', layout=Layout(width…